# Cloud Cost Optimization for AI/ML

## Overview
ML workloads can be extremely expensive. A single large model training run can cost thousands of dollars. This notebook covers systematic strategies to reduce costs without sacrificing quality.

---

## 1. Cost Drivers in ML

| Category | Examples | Cost Driver |
|----------|----------|-------------|
| **Compute** | GPU training, inference endpoints | $/hour × hours |
| **Storage** | S3, GCS, Blob | $/GB/month |
| **Data Transfer** | Egress between regions | $/GB |
| **API Calls** | OpenAI, Bedrock, Vertex | $/1M tokens |
| **Managed Services** | SageMaker overhead | 2× raw EC2 price |

---

## 2. Compute Optimization

### Spot / Preemptible Instances
- **AWS Spot**: up to 90% cheaper than On-Demand
- **GCP Preemptible/Spot**: up to 91% cheaper
- **Azure Spot**: up to 90% cheaper
- Risk: can be terminated with 2-minute warning
- Mitigation: checkpointing every N steps

$$\text{Cost}_{spot} = \text{Cost}_{on-demand} \times (1 - \text{discount})$$
$$\text{Expected Cost} = \text{Cost}_{spot} + P(\text{interruption}) \times \text{Cost}_{restart}$$

### Savings Plans / Reserved Instances
- Commit to 1 or 3 years → 30-72% discount
- Best for steady-state inference workloads

### Right-Sizing
- Use smallest instance that meets latency SLA
- Profile GPU utilization: aim for >80%
- Use AWS Compute Optimizer / GCP Recommender

### Auto-Scaling
- Scale to zero when no traffic (serverless endpoints)
- Target tracking: maintain 70% CPU/GPU utilization
- Predictive scaling for known traffic patterns

---

## 3. LLM API Cost Optimization

### Token Cost Comparison (approx. 2024)
| Model | Input (per 1M) | Output (per 1M) |
|-------|---------------|----------------|
| GPT-4o | $2.50 | $10.00 |
| Claude 3.5 Sonnet | $3.00 | $15.00 |
| Claude 3.5 Haiku | $0.80 | $4.00 |
| Gemini 1.5 Flash | $0.075 | $0.30 |
| Llama 3 (self-hosted) | ~$0.10 | ~$0.10 |

### Strategies
1. **Prompt caching**: cache system prompts (Anthropic/OpenAI) → 90% cheaper for repeated context
2. **Batching**: batch API requests (50% cheaper on OpenAI Batch API)
3. **Model routing**: use cheap model for easy queries, expensive for hard ones
4. **Output length control**: set `max_tokens` aggressively
5. **Caching responses**: cache identical queries with Redis/DynamoDB

---

## 4. Model-Level Optimization

### Quantization
Reduce model precision to cut memory and compute:
- FP32 → FP16: **2× memory reduction**, ~same quality
- FP16 → INT8: **4× memory reduction**, small quality loss
- INT8 → INT4: **8× memory reduction**, moderate quality loss
- GPTQ, AWQ: post-training quantization for LLMs

$$\text{Memory}_{INT8} = \frac{\text{Memory}_{FP32}}{4}$$

### vLLM for Throughput
- PagedAttention: efficient KV-cache management
- Continuous batching: 10-50× throughput vs naive serving
- $$\text{Throughput} = \frac{\text{tokens/s}}{\text{GPU cost/s}} = \text{tokens per dollar}$$

### Speculative Decoding
- Draft model generates k tokens cheaply
- Target model verifies in parallel
- 2-3× speedup for same quality

---

## 5. Storage Cost Optimization

- **S3 Intelligent-Tiering**: auto-moves infrequent data to cheaper tiers
- **Lifecycle policies**: delete old experiment artifacts after N days
- **Compression**: gzip/parquet for training data (5-10× size reduction)
- **Deduplication**: don't store the same base model multiple times

---

## 6. Cost Monitoring

- **AWS**: Cost Explorer, Budgets alerts, Cost Anomaly Detection
- **GCP**: Cost Management, Budgets & Alerts, Billing reports
- **Azure**: Cost Management + Billing, Cost alerts
- **OpenAI**: Usage dashboard, per-key limits

---

## 7. FinOps for ML Teams

Financial Operations principles for ML:
1. **Visibility**: tag all resources with project/team/experiment
2. **Accountability**: per-team cost dashboards
3. **Optimization**: weekly cost review meetings
4. **Culture**: engineers see the cost of their experiments

In [1]:
import numpy as np

# ── LLM Cost Calculator ─────────────────────────────────────────
def calculate_llm_cost(input_tokens, output_tokens, model='gpt-4o'):
    """Calculate cost for LLM API calls."""
    pricing = {
        'gpt-4o':             {'input': 2.50,  'output': 10.00},
        'gpt-4o-mini':        {'input': 0.15,  'output': 0.60},
        'claude-3.5-sonnet':  {'input': 3.00,  'output': 15.00},
        'claude-3.5-haiku':   {'input': 0.80,  'output': 4.00},
        'gemini-1.5-pro':     {'input': 1.25,  'output': 5.00},
        'gemini-1.5-flash':   {'input': 0.075, 'output': 0.30},
        'llama-3-self-hosted':{'input': 0.10,  'output': 0.10},
    }
    p = pricing[model]
    cost = (input_tokens / 1e6) * p['input'] + (output_tokens / 1e6) * p['output']
    return cost

# Example: 1M queries/day, 500 input + 200 output tokens each
queries_per_day = 1_000_000
avg_input = 500
avg_output = 200

print("Daily cost comparison:")
print("-" * 50)
for model in ['gpt-4o', 'gpt-4o-mini', 'claude-3.5-sonnet', 'claude-3.5-haiku', 'gemini-1.5-flash']:
    total_input = queries_per_day * avg_input
    total_output = queries_per_day * avg_output
    daily = calculate_llm_cost(total_input, total_output, model)
    monthly = daily * 30
    print(f"{model:<25}: ${daily:>8.2f}/day  ${monthly:>10.2f}/month")

Daily cost comparison:
--------------------------------------------------
gpt-4o                   : $ 3250.00/day  $  97500.00/month
gpt-4o-mini              : $  195.00/day  $   5850.00/month
claude-3.5-sonnet        : $ 4500.00/day  $ 135000.00/month
claude-3.5-haiku         : $ 1200.00/day  $  36000.00/month
gemini-1.5-flash         : $   97.50/day  $   2925.00/month


In [2]:
# ── Spot Instance Savings Calculator ───────────────────────────
def spot_savings(on_demand_price, spot_price, training_hours, interruption_rate=0.05, restart_cost=1.0):
    """Calculate expected savings from spot instances."""
    on_demand_cost = on_demand_price * training_hours
    expected_interruptions = training_hours * interruption_rate
    spot_cost = spot_price * training_hours + expected_interruptions * restart_cost
    savings = on_demand_cost - spot_cost
    savings_pct = savings / on_demand_cost * 100
    return {'on_demand': on_demand_cost, 'spot': spot_cost, 'savings': savings, 'savings_pct': savings_pct}

# p3.2xlarge: $3.06 on-demand, ~$0.91 spot
result = spot_savings(on_demand_price=3.06, spot_price=0.91, training_hours=100)
print(f"On-Demand cost: ${result['on_demand']:.2f}")
print(f"Spot cost:      ${result['spot']:.2f}")
print(f"Savings:        ${result['savings']:.2f} ({result['savings_pct']:.1f}%)")

On-Demand cost: $306.00
Spot cost:      $96.00
Savings:        $210.00 (68.6%)


In [3]:
# ── Simple Response Cache with TTL ─────────────────────────────
import hashlib, time
from functools import wraps

class LLMCache:
    """Simple in-memory cache for LLM responses."""
    def __init__(self, ttl=3600):
        self.cache = {}
        self.ttl = ttl
        self.hits = 0
        self.misses = 0

    def _key(self, prompt, model):
        return hashlib.sha256(f"{model}:{prompt}".encode()).hexdigest()

    def get(self, prompt, model):
        key = self._key(prompt, model)
        if key in self.cache:
            entry = self.cache[key]
            if time.time() - entry['ts'] < self.ttl:
                self.hits += 1
                return entry['response']
        self.misses += 1
        return None

    def set(self, prompt, model, response):
        key = self._key(prompt, model)
        self.cache[key] = {'response': response, 'ts': time.time()}

    @property
    def hit_rate(self):
        total = self.hits + self.misses
        return self.hits / total if total > 0 else 0

cache = LLMCache(ttl=3600)

def cached_llm_call(prompt, model='gpt-4o'):
    cached = cache.get(prompt, model)
    if cached:
        return cached, True  # cache hit
    # response = actual_llm_call(prompt, model)
    response = f"[Simulated response to: {prompt[:30]}...]"
    cache.set(prompt, model, response)
    return response, False  # cache miss

# Test
for prompt in ['What is RAG?', 'What is RAG?', 'Explain LLMs', 'What is RAG?']:
    resp, hit = cached_llm_call(prompt)
    print(f"{'HIT ' if hit else 'MISS'}: {prompt}")

print(f"\nCache hit rate: {cache.hit_rate:.1%}")

MISS: What is RAG?
HIT : What is RAG?
MISS: Explain LLMs
HIT : What is RAG?

Cache hit rate: 50.0%


## Additional Learning Resources

### Tools
- [FinOps Foundation](https://www.finops.org/)
- [AWS Cost Explorer](https://aws.amazon.com/aws-cost-management/aws-cost-explorer/)
- [GCP Pricing Calculator](https://cloud.google.com/products/calculator)
- [Azure Pricing Calculator](https://azure.microsoft.com/en-us/pricing/calculator/)
- [Artificial Analysis LLM Benchmarks & Pricing](https://artificialanalysis.ai/)

### Papers & Blogs
- [vLLM: PagedAttention Paper](https://arxiv.org/abs/2309.06180)
- [Speculative Decoding Paper](https://arxiv.org/abs/2211.17192)
- [GPTQ Quantization Paper](https://arxiv.org/abs/2210.17323)
- [Chip Huyen LLM Cost Analysis](https://huyenchip.com/2023/04/11/llm-engineering.html)

### Books
- *Cloud FinOps* J.R. Storment & Mike Fuller (O'Reilly)